# Phase 2: 30-Day Unplanned Hospital Readmission Prediction & Leakage Audit

This notebook trains and evaluates **30-day unplanned hospital readmission prediction models** on the full MIMIC-IV dataset.

### Key Methodological Protocols Enforced:
1. **Living Cohort Only:** Excludes admissions where `hospital_expire_flag == 1` or `deathtime` / `dod` is non-null. A patient who dies during the index admission cannot be readmitted; treating them as negative readmission cases introduces severe target proxy bias.
2. **Strict 24-Hour Time Windowing (`admittime` anchor):** Features are restricted to the **first 24 hours of admission**, excluding full-stay counts, duration aggregates, lab trajectory metrics, care-unit transfers, and post-discharge ICD diagnosis codes.
3. **Zero Patient Overlap (`subject_id`):** Grouped patient splitting across Train (70%), Validation (15%), and Test (15%).
4. **LACE Clinical Score Comparison:** Non-ML baseline benchmark comparison against classic clinical risk scoring.
5. **Isotonic Calibration & Threshold Selection:** Probability calibration and screening threshold selection for ~80% target recall.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display as display

sys.path.append('..')
from src.models.readmission import ReadmissionModelPipeline
from src.models.evaluation import evaluate_binary_predictions, find_optimal_threshold
from src.utils.logger import get_logger

log = get_logger('notebook_readmission')
print('Readmission pipeline modules imported successfully.')

## 1. Class Imbalance Strategy Note

> **Imbalance Strategy Note:** We utilize cost-sensitive weighting (`scale_pos_weight` for XGBoost and `class_weight='balanced'` for LightGBM and Logistic Regression). Explicitly, **no SMOTE or synthetic oversampling** is applied. A 2025 MIMIC-IV heart-failure readmission study found that tabular tree models (XGBoost, LightGBM) perform superiorly on raw imbalance distributions without synthetic oversampling, with only deep neural architectures benefiting from synthetic resampling.

## 2. Expected Publication Benchmark Expectations

- **Published MIMIC-IV 30-Day Readmission AUROC Range:** **0.65 to 0.70** (well-tuned XGBoost/LightGBM with engineered 24h features hit ~0.68–0.70, outperforming the classic LACE score's 0.60–0.68 range).
- **Expected AUPRC:** **0.25 to 0.35** (against a ~10–12% base prevalence in the living cohort, representing a **2.0x–2.5x enrichment over random**).
- **Red Flag Rule:** Any AUROC $> 0.75$ indicates feature leakage or observation window overflow.

In [ ]:
# Execute full Phase 2 readmission pipeline
from run_readmission_pipeline import run as run_pipeline

run_pipeline()

## 3. Results Summary & Markdown Report

Below is the complete model comparison table generated across all models, LACE clinical baseline, and calibrated predictions:

In [ ]:
report_path = Path('../reports/tables/readmission_model_comparison.md')
if report_path.exists():
    display.display(display.Markdown(report_path.read_text()))
else:
    print('Report file not found yet.')